In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import math
import xlsxwriter
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
pd.set_option('display.max_rows', None)
pd.set_option("display.precision", 10)

In [ ]:
#---------------------------------------------------------------CV Stuff---------------------------------------------------------------#
############ FILL IN the 'PotentialConstant' below for CV Files (RHE Value) ############
PotentialConstant = 0.687
###########

Area = 2.75 * 2.75 * math.pi / 100
Beginning = 0.4
End = 0.2
DeltaV = Beginning - End

############ FILL IN the 'FileList' below with all CV file names comma-separated like current format ############
FileList = ['cv1 here.txt', 'cv2 here.txt', 'cv3 here.txt', 'cv after here.txt']
#All files will be plotted against each other
#Last two files will be plotted against each other
#Last two files will have the integral calculated
###########

Range = len(FileList)
DataFrameListBeforeTreatment = []
DataFrameList = []
ScanRateList = []
ForwardCapacitanceList = []
BackwardCapacitanceList = []
TotalChargeList = []
TotalCapacitanceList = []
CycleList = []
TableList = []

for file in FileList:
    df = pd.read_csv(file)

    for rowidx in range(df.shape[0]):
        if ''.join(x for x in df.iloc[rowidx, 0] if not x.isdigit() and x not in '.').strip() == "Scan Rate (V/s) =":
            ScanRateList.append(float(''.join(x for x in df.iloc[rowidx, 0] if x.isdigit() or x in '.')))
            break

    for rowidx in range(df.shape[0]):
        if ''.join(x for x in df.iloc[rowidx, 0] if not x.isdigit()).strip() == "Segment =":
            CycleList.append(float(''.join(x for x in df.iloc[rowidx, 0] if x.isdigit())))
            break

    CycleEndValue = df.iloc[-1, 0]
    for rowidx in range(df.shape[0] - 2, -1, -1):
        if df.iloc[rowidx, 0] == CycleEndValue:
            CycleBeginning = rowidx
            break
    df.drop(df.index[0:CycleBeginning - 1], inplace = True)
    df.reset_index(drop=True, inplace = True)
    df.iloc[:, 0] = pd.to_numeric(df.iloc[:, 0])
    df.iloc[:, 1] = pd.to_numeric(df.iloc[:, 1])
    df.columns = ['Potential/V', 'Current/I']
    DataFrameListBeforeTreatment.append(df.copy(deep = True))
    df.columns = ['Potential/V vs RHE', 'Current Density/mA/cm^2']
    df = df.astype({
        'Potential/V vs RHE': 'float64',
        'Current Density/mA/cm^2': 'float64'
    })
    df.iloc[:, 0] += PotentialConstant
    df.iloc[:, 0] = df.iloc[:, 0].round(3)
    df.iloc[:, 1] = df.iloc[:, 1] * 1000 / Area
    DataFrameList.append(df)

#Capacitance Calculation

for index, dataframe in enumerate(DataFrameList):
    forwardcapacitance = 0
    for rowidx in range(dataframe.shape[0]):
        if CycleList[index] % 2 == 0:
            if dataframe.iloc[rowidx, 0] <= Beginning:
                forwardcapacitance += (dataframe.iloc[rowidx - 2, 0] - dataframe.iloc[rowidx - 1, 0]) * (dataframe.iloc[rowidx - 2, 1] + dataframe.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[index])
                if dataframe.iloc[rowidx, 0] <= End:
                    break
        if CycleList[index] % 2 == 1:
            if dataframe.iloc[rowidx, 0] >= End:
                forwardcapacitance += (dataframe.iloc[rowidx - 2, 0] - dataframe.iloc[rowidx - 1, 0]) * (dataframe.iloc[rowidx - 2, 1] + dataframe.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[index])
                if dataframe.iloc[rowidx, 0] >= Beginning:
                    break
    ForwardCapacitanceList.append(forwardcapacitance)

for index, dataframe in enumerate(DataFrameList):
    backwardcapacitance = 0
    for rowidx in range(dataframe.shape[0] - 1, -1, -1):
        if CycleList[index] % 2 == 0:
            if dataframe.iloc[rowidx, 0] <= Beginning:
                backwardcapacitance += (dataframe.iloc[rowidx - 2, 0] - dataframe.iloc[rowidx - 1, 0]) * (dataframe.iloc[rowidx - 2, 1] + dataframe.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[index])
                if dataframe.iloc[rowidx, 0] <= End:
                    break
        if CycleList[index] % 2 == 1:
            if dataframe.iloc[rowidx, 0] >= End:
                backwardcapacitance += (dataframe.iloc[rowidx - 2, 0] - dataframe.iloc[rowidx - 1, 0]) * (dataframe.iloc[rowidx - 2, 1] + dataframe.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[index])
                if dataframe.iloc[rowidx, 0] >= Beginning:
                    break
    BackwardCapacitanceList.append(backwardcapacitance)

for count, idx in enumerate(range(Range)):
    TotalCapacitanceList.append(abs((ForwardCapacitanceList[idx] + BackwardCapacitanceList[idx]) / (2 * DeltaV)))
    TotalChargeList.append(abs(ForwardCapacitanceList[idx] + BackwardCapacitanceList[idx]))

#---------------------------------------------------------------Integral Calculation Portion---------------------------------------------------------------#
############ FILL IN starting and ending x-value of integration for the TOP (from left to right of x-axis) ############
TopStartingIntegrationXValue = 0.4
TopEndingIntegrationXValue = 0.8
############

TopCurveCalculation = []
TopLineCalculation = []
TopIntegralFinalValues = []

############ FILL IN starting and ending x-value of integration for the BOTTOM (from left to right of x-axis) ############
BottomStartingIntegrationXValue = 0.4
BottomEndingIntegrationXValue = 0.8
###########

BottomCurveCalculation = []
BottomLineCalculation = []
BottomIntegralFinalValues = []

#Top Integral Calculation
for idx in range(len(FileList) - 2, len(FileList)):
    TopCurveValue = 0
    TopLineValue = 0
    TopStartRowIndex = None
    TopEndRowIndex = None
    normaldf = DataFrameList[idx]
    modifieddf = normaldf.copy(deep = True)
    for rowidx in range(df.shape[0]):
        if (round(normaldf.iloc[rowidx, 0], 3) == TopStartingIntegrationXValue):
            TopStartRowIndex = rowidx
            break
    
    for rowidx in range(df.shape[0]):
        if (round(normaldf.iloc[rowidx, 0], 3) == TopEndingIntegrationXValue):
            TopEndRowIndex = rowidx
            break
    

    slope = (normaldf.iloc[TopEndRowIndex, 1] - normaldf.iloc[TopStartRowIndex, 1]) / (normaldf.iloc[TopEndRowIndex, 0] - normaldf.iloc[TopStartRowIndex, 0])
    yintercept = normaldf.iloc[TopEndRowIndex, 1] - (slope * normaldf.iloc[TopEndRowIndex, 0])
    for rowidx in range(TopStartRowIndex, TopEndRowIndex + 1):
        modifieddf.iloc[rowidx, 1] = (slope * normaldf.iloc[rowidx, 0]) + yintercept

    for rowidx in range(normaldf.shape[0]):
        if normaldf.iloc[rowidx, 0] >= TopStartingIntegrationXValue:
            TopCurveValue += (normaldf.iloc[rowidx - 2, 0] - normaldf.iloc[rowidx - 1, 0]) * (normaldf.iloc[rowidx - 2, 1] + normaldf.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[idx])
            if normaldf.iloc[rowidx, 0] >= TopEndingIntegrationXValue:
                break
    TopCurveCalculation.append(abs(TopCurveValue))

    for rowidx in range(modifieddf.shape[0]):
        if modifieddf.iloc[rowidx, 0] >= TopStartingIntegrationXValue:
            TopLineValue += (modifieddf.iloc[rowidx - 2, 0] - modifieddf.iloc[rowidx - 1, 0]) * (modifieddf.iloc[rowidx - 2, 1] + modifieddf.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[idx])
            if modifieddf.iloc[rowidx, 0] >= TopEndingIntegrationXValue:
                break
    TopLineCalculation.append(abs(TopLineValue))


#Bottom Integral Calculation
for idx in range(len(FileList) - 2, len(FileList)):
    BottomCurveValue = 0
    BottomLineValue = 0
    BottomStartRowIndex = None
    BottomEndRowIndex = None
    regulardf = DataFrameList[idx]
    altereddf = regulardf.copy(deep=True)

    for rowidx in range(normaldf.shape[0] - 1, -1, -1):
        if round(regulardf.iloc[rowidx, 0], 3) == BottomStartingIntegrationXValue:
            BottomStartRowIndex = rowidx
            break
    
    for rowidx in range(normaldf.shape[0] - 1, -1, -1):
        if round(regulardf.iloc[rowidx, 0], 3) == BottomEndingIntegrationXValue:
            BottomEndRowIndex = rowidx
            break

    otherslope = (regulardf.iloc[BottomEndRowIndex, 1] - regulardf.iloc[BottomStartRowIndex, 1]) / (regulardf.iloc[BottomEndRowIndex, 0] - regulardf.iloc[BottomStartRowIndex, 0])
    otheryintercept = regulardf.iloc[BottomEndRowIndex, 1] - (otherslope * regulardf.iloc[BottomEndRowIndex, 0])

    for rowidx in range(BottomStartRowIndex, BottomEndRowIndex - 1, -1):
        altereddf.iloc[rowidx, 1] = (otherslope * regulardf.iloc[rowidx, 0]) + otheryintercept

    for rowidx in range(regulardf.shape[0] - 1, -1, -1):
        if regulardf.iloc[rowidx, 0] >= BottomStartingIntegrationXValue:
            BottomCurveValue += (regulardf.iloc[rowidx - 2, 0] - regulardf.iloc[rowidx - 1, 0]) * (regulardf.iloc[rowidx - 2, 1] + regulardf.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[idx])
            if regulardf.iloc[rowidx, 0] >= BottomEndingIntegrationXValue:
                
                break
    BottomCurveCalculation.append(abs(BottomCurveValue))

    for rowidx in range(altereddf.shape[0] - 1, -1, -1):
        if altereddf.iloc[rowidx, 0] >= BottomStartingIntegrationXValue:
            BottomLineValue += (altereddf.iloc[rowidx - 2, 0] - altereddf.iloc[rowidx - 1, 0]) * (altereddf.iloc[rowidx - 2, 1] + altereddf.iloc[rowidx - 1, 1]) * Area / (2 * ScanRateList[idx])
            if altereddf.iloc[rowidx, 0] >= BottomEndingIntegrationXValue:
                break
    BottomLineCalculation.append(abs(BottomLineValue))
    
  

for idx in range(0, 2):
    TopIntegralFinalValues.append(abs(TopCurveCalculation[idx] - TopLineCalculation[idx]))
    BottomIntegralFinalValues.append(abs(BottomCurveCalculation[idx] - BottomLineCalculation[idx]))

In [121]:
#Outputting graphs and plots to Excel file
ExcelFileName = FileList[0][0:-14] + '.xlsx'
SheetName = 'CV Data'
writer = pd.ExcelWriter(ExcelFileName, engine='xlsxwriter')
col = 0
for idx in range(len(FileList)):
    DataFrameListBeforeTreatment[idx].to_excel(writer, sheet_name = SheetName, index = False, startrow = 1, startcol = col)
    col += len(DataFrameListBeforeTreatment[idx].columns)
    DataFrameList[idx].to_excel(writer, sheet_name = SheetName, index = False, startrow = 1, startcol = col)
    col += len(DataFrameListBeforeTreatment[idx].columns) + 2

Legend = []
for file in FileList:
    name = file[0:len(file) - 4]
    Legend.append(name)
worksheet = writer.sheets[SheetName]
column = 0
for idx in range(len(FileList)):
    worksheet.write_string(0, column, Legend[idx])
    column += len(DataFrameListBeforeTreatment[idx].columns) + len(DataFrameListBeforeTreatment[idx].columns) + 2

workbook = writer.book
chart = workbook.add_chart({'type': 'scatter', 'subtype': 'smooth'})
namecol = 0
categorycol = 2
valuescol = 3
ColorDict = {1: 'red', 2: 'blue', 3: 'purple', 4: 'green', 5: 'yellow', 6: 'orange', 7: 'black', 8: 'pink', 9: 'white', 10: 'gray'}
for idx in range(len(FileList)):
    chart.add_series({
        'name': [SheetName, 0, namecol, 0, namecol],
        'categories': [SheetName, 2, categorycol, 2 + len(DataFrameList[idx]) - 1, categorycol],
        'values': [SheetName, 2,  valuescol, 2 + len(DataFrameList[idx]) - 1, valuescol],
    })
    namecol += 6
    categorycol += 6
    valuescol += 6
chart.set_title({'name': 'All CVs'})
chart.set_x_axis({'min': 0, 'max': 1, 'name': 'Potential, V vs RHE'})
chart.set_y_axis({'name': 'Current Density, mA/cm^2'})
worksheet.insert_chart(0, col, chart)

othernamecol = col - 12
othercategorycol = col - 10
othervaluescol = col - 9
otherchart = workbook.add_chart({'type': 'scatter', 'subtype': 'smooth'})
for idx in range(len(FileList) - 2, len(FileList)):
    otherchart.add_series({
        'name': [SheetName, 0, othernamecol, 0, othernamecol],
        'categories': [SheetName, 2, othercategorycol, 2 + len(DataFrameList[idx]) - 1, othercategorycol],
        'values': [SheetName, 2,  othervaluescol, 2 + len(DataFrameList[idx]) - 1, othervaluescol],
    })
    othernamecol += 6
    othercategorycol += 6
    othervaluescol +=6
otherchart.set_title({'name': 'CV before and after ORR'})
otherchart.set_x_axis({'min': 0, 'max': 1, 'name': 'Potential, V vs RHE'})
otherchart.set_y_axis({'name': 'Current Density, mA/cm^2'})
worksheet.insert_chart(15, col, otherchart)

DeltaVList = []
TableHeader = []
for idx in range(len(FileList)):
    ForwardCapacitanceList[idx] = abs(ForwardCapacitanceList[idx])
    BackwardCapacitanceList[idx] = abs(BackwardCapacitanceList[idx])
    DeltaVList.append(DeltaV)
    TableHeader.append(Legend[idx][32:])
TableList = [ScanRateList, DeltaVList, ForwardCapacitanceList, BackwardCapacitanceList, TotalChargeList, TotalCapacitanceList]
dftable = pd.DataFrame(TableList)
dftable.columns = TableHeader
dftable.index = ['Scan Rate (V/s)', 'Delta V (V)', 'Double Layer Charge Pos (mA)', 'Double Layer Charge Neg (mA)', 'Total Charge (mA)', 'Capacitance (mF)']
dftable.to_excel(writer, sheet_name = SheetName, startrow = 30, startcol = col)

OtherTableHeader = []
for idx in range(len(FileList) - 2, len(FileList)):
    OtherTableHeader.append(Legend[idx][32:])
OtherTableList = [TopIntegralFinalValues, BottomIntegralFinalValues]
otherdftable = pd.DataFrame(OtherTableList)
otherdftable.columns = OtherTableHeader
otherdftable.index = ['Top Charge (mCs)', 'Bottom Charge (mCs)']
otherdftable.to_excel(writer, sheet_name = SheetName, startrow = 38, startcol = col)


0

0

0

0

0

0

In [ ]:
#---------------------------------------------------------------ORR Stuff---------------------------------------------------------------#
############FILL IN the 'FileList' below with all ORR file names comma-separated with same format as CV FileList############
FileList = ['Fe2O3_ZIF8SA_2hBM_082024 SCV ORR.txt']
############

############ FILL IN 'RHEList' below with comma-separated RHE values for each ORR file in the respective order above ############
RHEList = [0.687]
############

Area = 2.75 * 2.75 * math.pi / 100

############ FILL IN changes to these variables if needed ############
CatalystLoading = 0.14255
RotationRate = 900
Efficiency = 0.378
############

CSVNumberOfColumns = 3 

#2D Arrays that correspond to a file with mutiple dataframes
AllDataFrameListBeforeTreatment = [] 
AllDataFrameListAfterTreatment = []
AllDataFrameListAverageTreatment = []
AllCombinedDataFrames = []

NumberOfCyclesList = []

for count, file in enumerate(FileList):
    DataFrameListPerFileBeforeTreatment = []
    DataFrameListPerFileAfterTreatment = []
    DataFrameListPerFileAverageTreatment = []
    df = pd.read_csv(file, names = list(range(0, CSVNumberOfColumns)))
    
    for rowidx in range(df.shape[0]):
        if ''.join(x for x in df.iloc[rowidx, 0] if not x.isdigit()).strip() == "Segment =":
            NumberOfCyclesList.append(float(''.join(x for x in df.iloc[rowidx, 0] if x.isdigit())) / 2)
            break
    
    for rowidx in range(df.shape[0]):
        if df.iloc[rowidx, 0].strip() == "Potential/V":
            df.drop(df.index[0:rowidx + 1], inplace = True)
            break
    
    EndingCycleValue = df.iloc[1, 0]
    for iteration in range(int(NumberOfCyclesList[count])):
        for rowidx in range(2, df.shape[0]):
            if df.iloc[rowidx, 0] == EndingCycleValue:
                untreateddf = df.iloc[0:rowidx + 1,:].copy(deep = True)
                untreateddf.iloc[:, 0] = pd.to_numeric(untreateddf.iloc[:, 0], errors='coerce')
                untreateddf.iloc[:, 1] = pd.to_numeric(untreateddf.iloc[:, 1], errors='coerce')
                untreateddf.iloc[:, 2] = pd.to_numeric(untreateddf.iloc[:, 2], errors='coerce')
                untreateddf.reset_index(drop=True, inplace = True)
                untreateddf[untreateddf.columns] = untreateddf[untreateddf.columns].astype('float64')
                DataFrameListPerFileBeforeTreatment.append(untreateddf)
                treateddf = untreateddf.copy(deep = True)
                treateddf.iloc[:, 2] = -200 * (treateddf.iloc[:, 2] / Efficiency) / (treateddf.iloc[:, 1] + treateddf.iloc[:, 2] / Efficiency)
                treateddf.iloc[:, 0] += RHEList[count]
                treateddf.iloc[:, 0] = treateddf.iloc[:, 0].round(3)
                treateddf.iloc[:, 1] = treateddf.iloc[:, 1] * 1000 / Area
                DataFrameListPerFileAfterTreatment.append(treateddf)
                df.drop(df.index[0:rowidx + 1], inplace = True)
                df.reset_index(drop=True, inplace = True)
                break
    AllDataFrameListBeforeTreatment.append(DataFrameListPerFileBeforeTreatment)
    AllDataFrameListAfterTreatment.append(DataFrameListPerFileAfterTreatment)

    for iteration in range(int(NumberOfCyclesList[count])):
        VvsRHEAverage = []
        CurrentDensityAverage = []
        Y_H2O2Average = []
        simplifyname = AllDataFrameListAfterTreatment[count][iteration]
        EndRowIndex = simplifyname.shape[0] - 1
        for rowidx in range(1, int(simplifyname.shape[0] / 2) + 1):
            VvsRHEAverage.append((simplifyname.iloc[rowidx, 0] + simplifyname.iloc[EndRowIndex, 0]) / 2)
            CurrentDensityAverage.append((simplifyname.iloc[rowidx, 1] + simplifyname.iloc[EndRowIndex, 1]) / 2)
            Y_H2O2Average.append((simplifyname.iloc[rowidx, 2] + simplifyname.iloc[EndRowIndex, 2]) / 2)
            EndRowIndex -= 1
        averagedf = pd.DataFrame(list(zip(VvsRHEAverage, CurrentDensityAverage, Y_H2O2Average)))
        DataFrameListPerFileAverageTreatment.append(averagedf)
    AllDataFrameListAverageTreatment.append(DataFrameListPerFileAverageTreatment)

Header = ['Potential/V', 'i1/A', 'i2/A', 'V vs RHE', 'i1 (mA/cm^2)', 'Y_H2O2', 'Average V vs RHE', 'Average i1 (mA/cm^2)', 'Average Y_H2O2']
for idx in range(len(FileList)):
    ConcatenatedDataFrames = []
    for count in range(int(NumberOfCyclesList[idx])):
        concatenateddf = pd.concat([AllDataFrameListBeforeTreatment[idx][count], AllDataFrameListAfterTreatment[idx][count], AllDataFrameListAverageTreatment[idx][count]], axis = 1)
        concatenateddf.columns = Header
        ConcatenatedDataFrames.append(concatenateddf)
    AllCombinedDataFrames.append(ConcatenatedDataFrames)

for count in range(len(FileList)):
    col = 0
    SheetName = 'ORR Data File ' + str(count + 1)
    for idx in range(int(NumberOfCyclesList[count])):
        AllCombinedDataFrames[count][idx].to_excel(writer, sheet_name = SheetName, index = False, startrow = 2, startcol = col)
        col += len(AllCombinedDataFrames[count][idx].columns) + 3
        
for count in range(len(FileList)):
    col = 0
    worksheet = writer.sheets['ORR Data File ' + str(count + 1)]
    worksheet.write_string(0, col, 'Filename: ' + FileList[count][0:-4])
    for idx in range(int(NumberOfCyclesList[count])):
        worksheet.write_string(1, col, FileList[count][0:-4] + str(idx + 1))
        col += len(AllCombinedDataFrames[count][idx].columns) + 3

for count in range(len(FileList)):
    namecol = 0
    categorycol = 3
    valuescol = 5
    workbook = writer.book
    worksheet = writer.sheets['ORR Data File ' + str(count + 1)]
    chart = workbook.add_chart({'type': 'scatter', 'subtype': 'smooth'})
    for idx in range(int(NumberOfCyclesList[count])):
        chart.add_series({
            'name': ['ORR Data File ' + str(count + 1), 1, namecol, 1, namecol],
            'categories': ['ORR Data File ' + str(count + 1), 3, categorycol, 3 + AllCombinedDataFrames[count][idx].iloc[:,3].size - 1, categorycol],
            'values': ['ORR Data File ' + str(count + 1), 3,  valuescol, 3 + AllCombinedDataFrames[count][idx].iloc[:,3].size - 1, valuescol],
        })
        namecol += 12
        categorycol += 12
        valuescol += 12
    chart.set_x_axis({'min': 0, 'max': 1, 'name': 'Potential (V vs RHE)'})
    chart.set_y_axis({'name': 'H2O2 (%)'})
    worksheet.insert_chart(AllCombinedDataFrames[0][0].shape[0] + 4, 0, chart)

for count in range(len(FileList)):
    namecol = 0
    categorycol = 3
    valuescol = 4
    workbook = writer.book
    worksheet = writer.sheets['ORR Data File ' + str(count + 1)]
    chart = workbook.add_chart({'type': 'scatter', 'subtype': 'smooth'})
    for idx in range(int(NumberOfCyclesList[count])):
        chart.add_series({
            'name': ['ORR Data File ' + str(count + 1), 1, namecol, 1, namecol],
            'categories': ['ORR Data File ' + str(count + 1), 3, categorycol, 3 + AllCombinedDataFrames[count][idx].iloc[:,3].size - 1, categorycol],
            'values': ['ORR Data File ' + str(count + 1), 3,  valuescol, 3 + AllCombinedDataFrames[count][idx].iloc[:,3].size - 1, valuescol],
        })
        namecol += 12
        categorycol += 12
        valuescol += 12
    chart.set_x_axis({'min': 0, 'max': 1, 'name': 'Potential (V vs RHE)'})
    chart.set_y_axis({'name': 'Current Density (mA/cm^2)'})
    worksheet.insert_chart(AllCombinedDataFrames[0][0].shape[0] + 4, 16, chart)

for count in range(len(FileList)):
    namecol = 0
    categorycol = 6
    valuescol = 7
    workbook = writer.book
    worksheet = writer.sheets['ORR Data File ' + str(count + 1)]
    chart = workbook.add_chart({'type': 'scatter', 'subtype': 'smooth'})
    for idx in range(int(NumberOfCyclesList[count])):
        chart.add_series({
            'name': ['ORR Data File ' + str(count + 1), 1, namecol, 1, namecol],
            'categories': ['ORR Data File ' + str(count + 1), 3, categorycol, 3 + AllCombinedDataFrames[count][idx].iloc[:,6].count() - 1, categorycol],
            'values': ['ORR Data File ' + str(count + 1), 3,  valuescol, 3 + AllCombinedDataFrames[count][idx].iloc[:,6].count() - 1, valuescol],
        })
        namecol += 12
        categorycol += 12
        valuescol += 12
    chart.set_x_axis({'min': 0, 'max': 1, 'name': 'Potential (V vs RHE)'})
    chart.set_y_axis({'name': 'Current Density (mA/cm^2)'})
    chart.set_title({'name': 'Average'})
    worksheet.insert_chart(AllCombinedDataFrames[0][0].shape[0] + 4, 24, chart)

for count in range(len(FileList)):
    namecol = 0
    categorycol = 6
    valuescol = 8
    workbook = writer.book
    worksheet = writer.sheets['ORR Data File ' + str(count + 1)]
    chart = workbook.add_chart({'type': 'scatter', 'subtype': 'smooth'})
    for idx in range(int(NumberOfCyclesList[count])):
        chart.add_series({
            'name': ['ORR Data File ' + str(count + 1), 1, namecol, 1, namecol],
            'categories': ['ORR Data File ' + str(count + 1), 6, categorycol, 3 + AllCombinedDataFrames[count][idx].iloc[:,6].count() - 6, categorycol],
            'values': ['ORR Data File ' + str(count + 1), 6,  valuescol, 3 + AllCombinedDataFrames[count][idx].iloc[:,6].count() - 6, valuescol],
        })
        namecol += 12
        categorycol += 12
        valuescol += 12
    chart.set_x_axis({'min': 0, 'max': 1, 'name': 'Potential (V vs RHE)'})
    chart.set_y_axis({'name': 'H2O2 (%)'})
    chart.set_title({'name': 'Average'})
    worksheet.insert_chart(AllCombinedDataFrames[0][0].shape[0] + 4, 8, chart)

#Rounding Row Float Values (Potential)
for count in range(len(FileList)):
    for idx in range(int(NumberOfCyclesList[count])):
        AllCombinedDataFrames[count][idx].iloc[:, 0] = AllCombinedDataFrames[count][idx].iloc[:, 0].apply(lambda x: round(x, 3))
        AllCombinedDataFrames[count][idx].iloc[:, 3] = AllCombinedDataFrames[count][idx].iloc[:, 3].apply(lambda x: round(x, 3))
        AllCombinedDataFrames[count][idx].iloc[:, 6] = AllCombinedDataFrames[count][idx].iloc[:, 6].apply(lambda x: round(x, 3))

#ORR Table Stuff
LimitingCurrentList = []
CurrentAt08VList = []
KineticCurrentList = []
MassActivityList = []
LimitingCurrentReciprocalList = []
CurrentAt08VReciprocalList = []
RPMRaisedNegativeHalfList = []
RPMList = []
H2O2YieldList = []


HalfofLimitingCurrentList = []
SlopeList = []
InterceptList = []
HalfPotentialList = []

AllCompletedTablesList = []
for count in range (len(FileList)):

    TempLimitingCurrentList = []
    TempCurrentAt08VList = []
    TempKineticCurrentList = []
    TempMassActivityList = []
    TempLimitngCurrentReciprocalList = []
    TempCurrentAt08VReciprocalList = []
    TempRPMRaisedNegativeHalfList = []
    TempRPMList = []
    TempHalfofLimitingCurrentList = []
    TempSlopeList = []
    TempInterceptList = []
    TempHalfPotentialList = []
    TempH2O2YieldList = []

    for idx in range(int(NumberOfCyclesList[count])):

        LimitingCurrent = 0
        CurrentAt08V = 0
        PointCount = 0
        H2O2Yield = 0

        Slope = 0
        Intercept = 0
        HalfPotential = 0

        CurrentDataFrame = AllCombinedDataFrames[count][idx]
        for rowidx in range(CurrentDataFrame.iloc[:,6].count()):
            if (round(CurrentDataFrame.iloc[rowidx, 6], 3) == 0.4):
                H2O2Yield = CurrentDataFrame.iloc[rowidx, 8]
            if (round(CurrentDataFrame.iloc[rowidx, 6], 3) == 0.8):
                CurrentAt08V = CurrentDataFrame.iloc[rowidx, 7]
            if (round(CurrentDataFrame.iloc[rowidx, 6], 3) <= 0.4) and (round(CurrentDataFrame.iloc[rowidx, 6], 3) >= 0.1):
                LimitingCurrent += CurrentDataFrame.iloc[rowidx, 7]
                PointCount += 1
        TempLimitingCurrentList.append(LimitingCurrent/PointCount)
        TempCurrentAt08VList.append(CurrentAt08V)
        TempKineticCurrentList.append(TempLimitingCurrentList[idx] * TempCurrentAt08VList[idx] / (TempLimitingCurrentList[idx] - TempCurrentAt08VList[idx]))
        TempMassActivityList.append(TempKineticCurrentList[idx] * Area / CatalystLoading)
        TempLimitngCurrentReciprocalList.append(1 / (TempLimitingCurrentList[idx] * Area))
        TempCurrentAt08VReciprocalList.append(1 / (TempCurrentAt08VList[idx] * Area))
        TempRPMRaisedNegativeHalfList.append(pow(RotationRate, -0.5))
        TempRPMList.append(RotationRate)
        TempHalfofLimitingCurrentList.append(TempLimitingCurrentList[idx] / 2)
        TempH2O2YieldList.append(H2O2Yield)


    

        for rowidx in range(1, CurrentDataFrame.iloc[:, 6].count()):
            if CurrentDataFrame.iloc[rowidx, 7] <= TempHalfofLimitingCurrentList[idx]:
                Slope = (CurrentDataFrame.iloc[rowidx - 1, 6] - CurrentDataFrame.iloc[rowidx, 6]) / (CurrentDataFrame.iloc[rowidx - 1, 7] - CurrentDataFrame.iloc[rowidx, 7])
                Intercept = CurrentDataFrame.iloc[rowidx, 6] - (Slope * CurrentDataFrame.iloc[rowidx, 7])
                HalfPotential = Intercept - Slope * abs(TempHalfofLimitingCurrentList[idx])
                break
        TempSlopeList.append(Slope)
        TempInterceptList.append(Intercept)
        TempHalfPotentialList.append(HalfPotential)

    LimitingCurrentList.append(TempLimitingCurrentList)
    CurrentAt08VList.append(TempCurrentAt08VList)
    KineticCurrentList.append(TempKineticCurrentList)
    MassActivityList.append(TempMassActivityList)
    LimitingCurrentReciprocalList.append(TempLimitngCurrentReciprocalList)
    CurrentAt08VReciprocalList.append(TempCurrentAt08VReciprocalList)
    RPMRaisedNegativeHalfList.append(TempRPMRaisedNegativeHalfList)
    RPMList.append(TempRPMList)
    HalfofLimitingCurrentList.append(TempHalfofLimitingCurrentList)
    SlopeList.append(TempSlopeList)
    InterceptList.append(TempInterceptList)
    HalfPotentialList.append(TempHalfPotentialList)
    H2O2YieldList.append(TempH2O2YieldList)

#Rounding Table Values
for count in range(len(FileList)):
    Tabledf = pd.DataFrame(list(zip(list(np.around(np.array(MassActivityList[count]), 2)), list(np.around(np.array(HalfPotentialList[count]), 3)), list(np.around(np.array(H2O2YieldList[count]), 2)), 
    list(np.around(np.array(LimitingCurrentList[count]), 2)), list(np.around(np.array(CurrentAt08VList[count]), 2)), list(np.around(np.array(KineticCurrentList[count]), 2)), 
    list(np.around(np.array(RPMList[count]), 2)), list(np.around(np.array(LimitingCurrentReciprocalList[count]), 2)), list(np.around(np.array(CurrentAt08VReciprocalList[count]), 2)), 
    list(np.around(np.array(RPMRaisedNegativeHalfList[count]), 2)), list(np.around(np.array(HalfofLimitingCurrentList[count]), 2)), list(np.around(np.array(SlopeList[count]), 2)), 
    list(np.around(np.array(InterceptList[count]), 2))))).T.abs()

    AllCompletedTablesList.append(Tabledf)


CompletedTableIndex = ['Mass activity (mA/mg, at 0.8 V vs RHE)', 'E1/2 (V vs RHE)', 'H2O2 yield (%, at 0.4 V vs RHE)', 'i_lim (mA/cm^2)', 'i_at 0.8V (mA/cm^2)', 'i_kin (mA/cm^2)', 'rpm', '1/i_lim', 
'1/i_at 0.8V', 'rpm^-(1/2)', '1/2 i_lim', 'slope', 'intercept']

#Outputting to ExcelFile
for count in range(len(FileList)):
    CompletedTableHeader = []
    DFToExcel = AllCompletedTablesList[count]
    SheetName = 'ORR Data File ' + str(count + 1)
    for idx in range(int(NumberOfCyclesList[count])):
        CompletedTableHeader.append(FileList[count][0:-4] + str(idx + 1))
    DFToExcel.columns = CompletedTableHeader
    DFToExcel.index = CompletedTableIndex
    DFToExcel.to_excel(writer, sheet_name = SheetName, index = True, startrow = AllCombinedDataFrames[0][0].shape[0] + 19, startcol = 0)

        
writer.close()

0

0

0

0

0

0

0

0

0

0